# 40 — Regression + Classification (§9.1/§9.2)

Runs both regression (`azlib.models.fit_regression`, kinds: linear/poly2/gbr)
and classification (`azlib.models.fit_classification`, kinds:
logistic/gbc) for every same-space 1D/2D/3D attribute group of every
indicator (`azlib.indicators.INDICATORS`/`ATTRS`/`azlib.models.groups`) —
never cross-indicator (design spec §9). 1D/2D groups also get a saved chart
(`azlib.models.plot_1d`/`plot_2d`); 3D is report-only (no plot — matches
`plot_1d`/`plot_2d`'s own module docstring: "3D is not plotted").

Regression target: `label_coeff` (`azlib.space.label_coeff`, continuous,
`[0, 1]`). Classification target: the SAME `label_coeff`, thresholded at
`>= 0.5` into two classes — the identical split `azlib.models.plot_1d`/
`plot_2d` already use for point coloring (see that module's docstring); no
separate classification target is defined anywhere else in `azlib`, so this
notebook reuses that one documented split rather than inventing a second
convention. Indicator attributes are z-scored via TRAIN-frozen stats
(`azlib.indicators.fit_stats`, fit on this SAME dataset — this is an
exploratory single-dataset notebook, not a train/OOS split; see
`90_validate.ipynb` for that) before fitting, matching
`azlib.validate.run_train`'s own normalization convention.

Groups with fewer than `_MIN_FIT_POINTS` usable (non-NaN) rows are skipped
(same guard `azlib.validate._fit_all_groups` uses); a group whose derived
classification label is single-class after masking skips classification
only (regression is unaffected — it does not need class balance).

In [ ]:
tf = 15
direction = "long"

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebooks/action_zones/{azlib,nb} onto sys.path. Assumes cwd is
# the repo root (/code) -- true for the experiment Docker service
# (docker-compose.experiment.yml's working_dir: /code; papermill's own
# cwd=None default inherits the invoking process's cwd) and for the
# automated smoke test (tests/test_layer9_notebooks.py sets the papermill
# subprocess's cwd to the repo root to match). helpers.py/indicators.*
# resolve for free via the image's own PYTHONPATH=/code.
_AZ_ROOT = Path.cwd() / "notebooks" / "action_zones"
for _p in (_AZ_ROOT, _AZ_ROOT / "nb"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import nbutil

In [ ]:
import numpy as np

from azlib.loader import load_wide_df
from azlib.space import label_coeff
from azlib.indicators import INDICATORS, fit_stats, attribute_frame
from azlib.models import (
    REG_KINDS, CLF_KINDS, groups,
    fit_regression, fit_classification,
    plot_1d, plot_2d, save_result,
)

In [ ]:
wide_df = load_wide_df()
stats = fit_stats(wide_df, tf)
y_coeff = label_coeff(wide_df, tf, direction).to_numpy(dtype=float)
y_label = (y_coeff >= 0.5).astype(int)  # see markdown cell above for this choice
out_dir = nbutil.results_dir(tf, direction, "reg_clf")
print(f"wide_df: {len(wide_df)} rows -> out_dir={out_dir}")

In [ ]:
_MIN_FIT_POINTS = 10  # matches azlib.validate's own guard against degenerate sklearn fits

records = []

for indicator in INDICATORS:
    af = attribute_frame(wide_df, tf, indicator, stats)
    for dim in (1, 2, 3):
        for group in groups(indicator, dim):
            group_name = "-".join(group)
            X_full = af[list(group)].to_numpy(dtype=float)
            mask = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_coeff)
            n = int(mask.sum())
            if n < _MIN_FIT_POINTS:
                records.append({"indicator": indicator, "group": group_name, "dim": dim, "n": n, "skipped": True})
                continue

            X = X_full[mask]
            y_reg = y_coeff[mask]
            y_clf = y_label[mask]
            has_both_classes = len(np.unique(y_clf)) >= 2

            for kind in REG_KINDS:
                res = fit_regression(X, y_reg, kind=kind)
                path = f"{out_dir}/{indicator}_{group_name}_{kind}_reg.json"
                save_result(res, path)
                rec = {
                    "indicator": indicator, "group": group_name, "dim": dim, "n": n,
                    "family": "reg", "kind": kind, "metrics": res.metrics, "path": path,
                }
                if dim == 1:
                    chart_path = f"{out_dir}/{indicator}_{group_name}_{kind}_reg.png"
                    plot_1d(X[:, 0], y_reg, res, chart_path)
                    rec["chart"] = chart_path
                elif dim == 2:
                    chart_path = f"{out_dir}/{indicator}_{group_name}_{kind}_reg.png"
                    plot_2d(X[:, 0], X[:, 1], y_reg, res, chart_path)
                    rec["chart"] = chart_path
                records.append(rec)

            if not has_both_classes:
                records.append({
                    "indicator": indicator, "group": group_name, "dim": dim, "n": n,
                    "family": "clf", "skipped": True, "reason": "single-class label",
                })
                continue

            for kind in CLF_KINDS:
                res = fit_classification(X, y_clf, kind=kind)
                path = f"{out_dir}/{indicator}_{group_name}_{kind}_clf.json"
                save_result(res, path)
                rec = {
                    "indicator": indicator, "group": group_name, "dim": dim, "n": n,
                    "family": "clf", "kind": kind, "metrics": res.metrics, "path": path,
                }
                if dim == 1:
                    chart_path = f"{out_dir}/{indicator}_{group_name}_{kind}_clf.png"
                    plot_1d(X[:, 0], y_reg, res, chart_path)
                    rec["chart"] = chart_path
                elif dim == 2:
                    chart_path = f"{out_dir}/{indicator}_{group_name}_{kind}_clf.png"
                    plot_2d(X[:, 0], X[:, 1], y_reg, res, chart_path)
                    rec["chart"] = chart_path
                records.append(rec)

print(f"{len(records)} model records")

In [ ]:
report = {
    "tf": tf,
    "direction": direction,
    "n_rows": len(wide_df),
    "n_records": len(records),
    "records": records,
}
report_path = nbutil.write_json_report(f"{out_dir}/reg_clf_report.json", report)
print(report_path)